## Circle Area Analysis Workflow - ArcGIS Pro Python Notebook

In [ ]:
import arcpy    

# Input data paths (adjust as needed)
areas_csv = r"C:\data\areas.csv"
cities_csv = r"C:\data\cities.csv"
gdb_path = r"C:\temp\circle_stats.gdb"

In [ ]:
# 1. Create point layers from CSV tables
arcpy.management.XYTableToPoint(
    input_table=areas_csv,
    output_feature_class=f"{gdb_path}/areas.shp",
    x_field="longitude",
    y_field="latitude",
    coordinate_system="4326"
)

arcpy.management.XYTableToPoint(
    input_table=cities_csv,
    output_feature_class=f"{gdb_path}/cities.shp",
    x_field="longitude",
    y_field="latitude",
    coordinate_system="4326"
)

In [ ]:
# 2. Buffer areas using distance column (assumes field name "distance_m")
arcpy.analysis.Buffer(
    in_features=r"C:\temp\circle_stats.gdb/areas.shp",
    out_feature_class=f"{gdb_path}/area_buffers.shp",
    distance=arcpy.Field(100),  # replace with your actual value or use a field
    dissolve_option="NONE",     # keep one buffer per row
    method="PLANAR"             # automatic geodesic for geographic input
)            

In [ ]:
# 3. Summarize within: count cities + sum population
summary_fields = [["population", "Sum"]]

arcpy.analysis.SummarizeWithin(
    in_polygons=f"{gdb_path}/area_buffers.shp",
    in_summary_features=r"C:\temp\circle_stats.gdb/cities.shp",
    out_feature_class=f"{gdb_path}/summary.csv",
    summary_fields=summary_fields,
    keep_all_input_polygons=True
)

## Key points

- All output is stored locally in circle_stats.gdb
- Use consistent X/Y field names between CSVs (longitude/latitude)
- Field "distance_m" must exist in your circles CSV with meter values
- Summary CSV will contain: circle identifier, city count, total population
- No temporary files are created beyond the GDB path you specify.